# Analyze a v3 load test: user ramp, think times and parallel models

This notebook only **reads** the result files written by `RunLoadTest_v3` and `RunPerfScenario_v3` under
`Files/PerfScenarios/logs/` of its default lakehouse. It does not start any test.

Every run leaves a folder `<test_name_prefix>-<model_key>-c<N>-<timestamp>/` with one CSV per virtual user and, when think
times were enabled, a `thinktimes/` sub-folder. The CSV files are read with DuckDB, without copying them into tables.

| Query | What it shows |
|---|---|
| 1 | One ramp level per model, with as many distinct users as `concurrent_threads` |
| 2 | The ramp is **sequential** inside each model (each level starts after the previous one ends) |
| 3 | Concurrency actually reached and start window (`ramp_up_sec`) |
| 4 | Every query of the JSON file was executed, grouped by action |
| 5 | Think times: recorded, scaled and applied pause (ratio = `think_time_scale`) |
| 6 | **Main proof**: the measured wait between actions matches the applied think time |
| 7 | One user, action by action |
| 8 | Query latency (mean and percentiles) per concurrency level |
| 9 | Models run in **parallel** (their runs overlap in time) |

Requirement: the default lakehouse of this notebook must be the one where the tests write their results.
If `import duckdb` fails, run `%pip install duckdb` first.

In [ ]:
import duckdb
from datetime import datetime

# --- Parameters ----------------------------------------------------------------------------------------
LOGS = "/lakehouse/default/Files/PerfScenarios/logs"   # default lakehouse of this notebook
RUN_PREFIX = "ramp-v3"      # = test_name_prefix of the pipeline
SINCE_UTC = None            # for example "2026-10-28T10:00:00Z" to ignore earlier runs

# --- Views over the CSV files ----------------------------------------------------------------------------
# Fixed dialect: the DAX text in the CSV is multi-line and quoted, and automatic detection can fail with it.
CSV_OPTS = "header = true, union_by_name = true, delim = ',', quote = '\"', escape = '\"', strict_mode = false"
since_filter = ""
if SINCE_UTC:
    since_epoch = datetime.fromisoformat(SINCE_UTC.replace("Z", "+00:00")).timestamp()
    since_filter = f"WHERE start_time >= {since_epoch}"

con = duckdb.connect()
con.execute("CREATE OR REPLACE MACRO ts(x) AS epoch_ms(CAST(x * 1000 AS BIGINT))")  # epoch seconds -> UTC TIMESTAMP
con.execute(f"""
    CREATE OR REPLACE VIEW results AS
    SELECT *, start_time + duration AS end_time
    FROM read_csv('{LOGS}/{RUN_PREFIX}-*/*_user_*.csv', {CSV_OPTS})
    {since_filter}
""")
if "status" not in {row[0] for row in con.execute("DESCRIBE results").fetchall()}:
    # runs written before version 3.4.0 have no status/error columns: every recorded query succeeded
    con.execute(f"""
        CREATE OR REPLACE VIEW results AS
        SELECT *, 'ok' AS status, CAST(NULL AS VARCHAR) AS error, start_time + duration AS end_time
        FROM read_csv('{LOGS}/{RUN_PREFIX}-*/*_user_*.csv', {CSV_OPTS})
        {since_filter}
    """)
con.execute("CREATE OR REPLACE MACRO ok(s) AS coalesce(s, 'ok') = 'ok'")  # a query that did not fail
try:
    con.execute(f"""
        CREATE OR REPLACE VIEW pauses AS
        SELECT * FROM read_csv('{LOGS}/{RUN_PREFIX}-*/thinktimes/*_thinktimes.csv', {CSV_OPTS})
        WHERE loadtest_id IN (SELECT loadtest_id FROM results)
    """)
except Exception as exc:
    print("Warning: no think-time CSV files found (the test ran with enable_think_times = false). "
          "Queries 5, 6 and 7 will not return rows.", type(exc).__name__)

runs, rows = con.execute("SELECT count(DISTINCT loadtest_id), count(*) FROM results").fetchone()
print(f"Runs found: {runs}  |  queries recorded: {rows}")
if runs == 0:
    raise ValueError(f"No results in {LOGS}/{RUN_PREFIX}-*. Check the prefix and that the default lakehouse is the one the tests write to.")


def q(sql):
    df = con.execute(sql).fetchdf()
    display(df)
    return df

In [ ]:
# 1. Runs recorded: one ramp level per model
# Each row is one ramp level of one model. 'distinct_users' must equal 'level'.
q("""
SELECT model,
       concurrent_threads AS level,
       count(DISTINCT thread_id) AS distinct_users,
       count(DISTINCT thread_id) = concurrent_threads AS users_ok,
       count(*) AS queries,
       count(*) FILTER (WHERE NOT ok(status)) AS failed_queries,
       ts(min(start_time)) AS start,
       ts(max(end_time)) AS "end",
       loadtest_id AS run_id
FROM results
GROUP BY model, concurrent_threads, loadtest_id
ORDER BY model, min(start_time)
""")

In [ ]:
# 2. The ramp is sequential inside each model (and models run in parallel)
# Each level must start after the previous level of the same model has ended ('OK: sequential').
q("""
WITH runs AS (
    SELECT model, concurrent_threads AS level, min(start_time) AS t0, max(end_time) AS t1
    FROM results GROUP BY model, concurrent_threads, loadtest_id
)
SELECT model,
       level,
       lead(level) OVER w AS next_level,
       ts(t0) AS start,
       ts(t1) AS "end",
       round(t1 - t0, 1) AS duration_s,
       round(lead(t0) OVER w - t1, 1) AS wait_until_next_level_s,
       CASE WHEN lead(t0) OVER w IS NULL THEN 'last level'
            WHEN lead(t0) OVER w >= t1 THEN 'OK: sequential'
            ELSE 'OVERLAPPED' END AS sequence
FROM runs
WINDOW w AS (PARTITION BY model ORDER BY t0)
ORDER BY model, t0
""")

In [ ]:
# 3. Concurrency actually reached and start window (ramp_up_sec)
# 'peak_concurrent' is the number of users active at the same time; 'start_window_s' is how long the last user took to start.
q("""
WITH u AS (
    SELECT loadtest_id, model, concurrent_threads AS level, thread_id,
           min(start_time) AS t0, max(end_time) AS t1
    FROM results GROUP BY loadtest_id, model, concurrent_threads, thread_id
),
ev AS (
    SELECT loadtest_id, t0 AS t, 1 AS d FROM u
    UNION ALL
    SELECT loadtest_id, t1, -1 FROM u
),
cum AS (
    SELECT loadtest_id,
           sum(d) OVER (PARTITION BY loadtest_id ORDER BY t, d DESC
                        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS active
    FROM ev
),
pk AS (SELECT loadtest_id, max(active) AS peak FROM cum GROUP BY loadtest_id)
SELECT u.model,
       u.level,
       count(*) AS users,
       any_value(pk.peak) AS peak_concurrent,
       round(max(u.t0) - min(u.t0), 1) AS start_window_s,
       round(avg(u.t1 - u.t0), 1) AS avg_duration_per_user_s
FROM u JOIN pk ON pk.loadtest_id = u.loadtest_id
GROUP BY u.model, u.level, u.loadtest_id
ORDER BY model, level
""")

In [ ]:
# 4. Every query of the JSON file was executed, grouped by action
# Each user and iteration must run all the queries of the JSON file (same count in every row).
q("""
SELECT model, level,
       min(queries) AS min_queries_per_user_iteration,
       max(queries) AS max_queries_per_user_iteration,
       min(actions) AS min_actions,
       max(actions) AS max_actions,
       count(*) AS user_iterations
FROM (
    SELECT model, concurrent_threads AS level, loadtest_id, thread_id, iteration,
           count(*) AS queries, count(DISTINCT action_index) AS actions
    FROM results GROUP BY model, concurrent_threads, loadtest_id, thread_id, iteration
)
GROUP BY model, level
ORDER BY model, level
""")

In [ ]:
# 5. Think times recorded: original, scaled and applied pause
# applied/original must be ~ think_time_scale (with jitter and cap). One pause per action.
q("""
SELECT loadtest_id AS run_id,
       count(DISTINCT thread_id) AS users,
       count(*) AS pauses,
       round(count(*) * 1.0 / count(DISTINCT CAST(thread_id AS VARCHAR) || '-' || CAST(iteration AS VARCHAR)), 1) AS pauses_per_user_iteration,
       max(think_time_scale) AS scale,
       round(sum(original_think_time_sec), 1) AS sum_original_s,
       round(sum(scaled_think_time_sec), 1) AS sum_scaled_s,
       round(sum(applied_think_time_sec), 1) AS sum_applied_s,
       round(sum(applied_think_time_sec) / nullif(sum(original_think_time_sec), 0), 3) AS applied_to_original_ratio,
       round(max(applied_think_time_sec), 2) AS max_pause_s
FROM pauses
WHERE source_label <> 'initial_offset'
GROUP BY loadtest_id
ORDER BY loadtest_id
""")

In [ ]:
# 6. MAIN PROOF: the measured wait between actions matches the applied think time
# actual pause = start of the action - end of the last query of the previous action (measured from the query results).
q("""
WITH act AS (
    SELECT loadtest_id, model, concurrent_threads AS level, thread_id, iteration, action_index,
           min(start_time) AS t_start, max(end_time) AS t_end,
           max(applied_think_time_sec) AS applied_pause
    FROM results
    GROUP BY loadtest_id, model, concurrent_threads, thread_id, iteration, action_index
),
gaps AS (
    SELECT *, t_start - lag(t_end) OVER (PARTITION BY loadtest_id, thread_id, iteration ORDER BY t_start) AS actual_pause
    FROM act
)
SELECT model, level,
       count(*) AS measured_actions,
       round(avg(applied_pause), 2) AS avg_applied_pause_s,
       round(avg(actual_pause), 2) AS avg_actual_pause_s,
       round(avg(abs(actual_pause - applied_pause)), 3) AS avg_deviation_s,
       round(max(abs(actual_pause - applied_pause)), 3) AS max_deviation_s
FROM gaps
WHERE actual_pause IS NOT NULL
GROUP BY model, level
ORDER BY model, level
""")

In [ ]:
# 7. One user, action by action (latest run, lowest user id, first iteration)
# Each action of the JSON file, how many queries it ran, the applied pause and the measured wait.
q("""
WITH last_run AS (
    SELECT loadtest_id FROM results GROUP BY loadtest_id ORDER BY min(start_time) DESC LIMIT 1
),
act AS (
    SELECT r.loadtest_id, r.thread_id, r.iteration, r.action_index,
           any_value(r.source_label) AS action,
           count(*) AS queries,
           min(r.start_time) AS t_start, max(r.end_time) AS t_end,
           max(r.original_think_time_sec) AS recorded_pause,
           max(r.applied_think_time_sec) AS applied_pause
    FROM results r JOIN last_run l ON l.loadtest_id = r.loadtest_id
    GROUP BY r.loadtest_id, r.thread_id, r.iteration, r.action_index
),
one AS (
    SELECT * FROM act
    WHERE thread_id = (SELECT min(thread_id) FROM act) AND iteration = (SELECT min(iteration) FROM act)
)
SELECT any_value(loadtest_id) OVER () AS run_id, action_index AS action_no, action, queries,
       round(t_start - min(t_start) OVER (), 2) AS relative_start_s,
       recorded_pause AS recorded_pause_s,
       applied_pause AS applied_pause_s,
       round(t_start - lag(t_end) OVER (ORDER BY t_start), 2) AS actual_pause_s
FROM one ORDER BY t_start
""")

In [ ]:
# 8. Query latency per concurrency level
# Effect of the ramp on model performance: mean and percentiles of the duration of the successful queries.
# Failed queries (timeouts, throttling, RLS errors...) are counted apart; their duration would distort the percentiles.
q("""
SELECT model, concurrent_threads AS level, count(*) AS queries,
       count(*) FILTER (WHERE NOT ok(status)) AS failed_queries,
       round(count(*) FILTER (WHERE NOT ok(status)) / count(*), 4) AS error_rate,
       round(avg(duration) FILTER (WHERE ok(status)), 3) AS mean_s,
       round(quantile_cont(duration, 0.5) FILTER (WHERE ok(status)), 3) AS p50_s,
       round(quantile_cont(duration, 0.9) FILTER (WHERE ok(status)), 3) AS p90_s,
       round(quantile_cont(duration, 0.95) FILTER (WHERE ok(status)), 3) AS p95_s,
       round(max(duration) FILTER (WHERE ok(status)), 3) AS max_s
FROM results
GROUP BY model, concurrent_threads
ORDER BY model, level
""")

In [ ]:
# 9. Models run in parallel (overlap of their runs)
# Pairs of models whose runs coincide in time; 'overlap_s' > 0 means simultaneous execution.
q("""
WITH runs AS (
    SELECT model, concurrent_threads AS level, min(start_time) AS t0, max(end_time) AS t1
    FROM results GROUP BY model, concurrent_threads, loadtest_id
)
SELECT a.model AS model_a, a.level AS level_a, b.model AS model_b, b.level AS level_b,
       round(a.t0 - b.t0, 2) AS start_difference_s,
       round(least(a.t1, b.t1) - greatest(a.t0, b.t0), 2) AS overlap_s
FROM runs a JOIN runs b ON a.model < b.model AND least(a.t1, b.t1) > greatest(a.t0, b.t0)
ORDER BY a.t0, model_a, model_b
""")